# Notebook 14 - Figures

Generates the results figures from the saved result files in `results/` and from the raw
laboratory data. Nothing here retrains or recomputes model output. Every number in every
figure traces to notebook 08 (seed 42, stratified folds, cross-fitted calibration) or to
the published Zagar and Mihelic (2022) dataset.

Figures are written to `results/figures/`, alongside fig1 to fig4 from the earlier EDA work.
File numbers are build order, not thesis figure numbers. The report assigns its own.

Style is fixed in 14.1 and applies to every figure: Agg backend, 150 dpi on screen, 300 dpi
on save, tight bounding box, 9 point font.

## 14.1 Setup

Resolves paths, fixes the style, loads the laboratory table and the result files, and
asserts the batch and cluster counts against the raw data before anything is plotted.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import GroupKFold

plt.rcParams["figure.dpi"]=150
plt.rcParams["savefig.dpi"]=300
plt.rcParams["savefig.bbox"]="tight"
plt.rcParams["font.size"]=9
plt.rcParams["grid.alpha"]=0.3

DATA=Path(r"C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets")
REPO=Path(r"C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis")
RESULTS=REPO/"results"
FIGURES=RESULTS/"figures"
assert DATA.exists()
assert RESULTS.exists()
assert FIGURES.exists()

lab=pd.read_csv(DATA/"Laboratory.csv",sep=";")
fold_assign=json.loads((RESULTS/"fold_assignment.json").read_text())
cluster=json.loads((RESULTS/"hardness_cluster_finding.json").read_text())
extended=json.loads((RESULTS/"extended_metrics.json").read_text())
mtj=json.loads((RESULTS/"mttrajnet_stratified.json").read_text())

HIGH=cluster["high_cluster_codes"]
OOD=fold_assign["ood_code"]
lab["is_high"]=lab["code"].isin(HIGH)

assert len(lab)==1005
assert lab["code"].nunique()==25
assert HIGH==[4,8,15,20,24]
assert OOD==15
assert int(lab["is_high"].sum())==cluster["high_cluster_batches"]==98
assert int((~lab["is_high"]).sum())==cluster["low_cluster_batches"]==907

print("lab",lab.shape,"high",int(lab["is_high"].sum()),"low",int((~lab["is_high"]).sum()))
print("high codes",HIGH,"ood code",OOD,"ood batches",int((lab["code"]==OOD).sum()))
print("loaded: fold_assignment, hardness_cluster_finding, extended_metrics, mttrajnet_stratified")

lab (1005, 56) high 98 low 907
high codes [4, 8, 15, 20, 24] ood code 15 ood batches 64
loaded: fold_assignment, hardness_cluster_finding, extended_metrics, mttrajnet_stratified


## 14.2 Figure 5 - fold diagnostic

The before panel is regenerated by running plain GroupKFold(3) on the 1005 batches grouped
by product code. It is not read from a file because the broken assignment was never saved as
a result. GroupKFold is deterministic, so the panel reproduces from the data rather than
being remembered.

The after panel is read from fold_assignment.json.

The figure plots high-cluster batches in the test fold against high-cluster batches in the
training folds, for each fold, because that contrast is the pathology. Under plain
GroupKFold fold 1 is tested on 90 of the 98 high-cluster batches while trained on 8, so the
test distribution cannot be learned from the training folds.

The after panel is not balanced either. Holding code 15 out leaves 34 high-cluster batches
split 22/6/6, so fold 1 is still tested on 22 while trained on 12. That residual imbalance is
shown, not hidden, and is why hardness RMSE stays elevated relative to the other targets.

In [2]:
gkf=GroupKFold(n_splits=3)
broken={}
for i,(tr,te) in enumerate(gkf.split(np.zeros(len(lab)),groups=lab["code"].values),1):
    broken[i]={"test_high":int(lab.iloc[te]["is_high"].sum()),
               "train_high":int(lab.iloc[tr]["is_high"].sum()),
               "test_n":int(len(te))}

strat={}
for i in (1,2,3):
    codes=fold_assign["folds"]["fold_%d"%i]
    sub=lab[lab["code"].isin(codes)]
    strat[i]={"test_high":int(sub["is_high"].sum()),"test_n":int(len(sub))}
tot_high=sum(strat[i]["test_high"] for i in (1,2,3))
for i in (1,2,3):
    strat[i]["train_high"]=tot_high-strat[i]["test_high"]
ood_n=int((lab["code"]==OOD).sum())

assert [broken[i]["test_high"] for i in (1,2,3)]==[90,6,2]
assert [strat[i]["test_high"] for i in (1,2,3)]==[22,6,6]
assert [strat[i]["test_high"] for i in (1,2,3)]==[fold_assign["high_cluster_per_fold"]["fold_%d"%i] for i in (1,2,3)]
assert sum(strat[i]["test_n"] for i in (1,2,3))==941
assert tot_high+ood_n==cluster["high_cluster_batches"]
print("checks passed")
for i in (1,2,3):
    print(" fold %d  broken test/train %3d/%3d   stratified test/train %3d/%3d"%(
        i,broken[i]["test_high"],broken[i]["train_high"],strat[i]["test_high"],strat[i]["train_high"]))

C_TEST="#c0392b"
C_TRAIN="#2e6da4"
x=np.arange(3)
w=0.38
fig,axes=plt.subplots(1,2,figsize=(9.2,3.8),sharey=True)

for ax,src,title in [(axes[0],broken,"(a) Plain GroupKFold(3), all 1005 batches"),
                     (axes[1],strat,"(b) Stratified assignment, code %d held out"%OOD)]:
    te=[src[i]["test_high"] for i in (1,2,3)]
    tr=[src[i]["train_high"] for i in (1,2,3)]
    b1=ax.bar(x-w/2,te,w,color=C_TEST,label="in test fold")
    b2=ax.bar(x+w/2,tr,w,color=C_TRAIN,label="in training folds")
    ax.bar_label(b1,fontsize=8,padding=2)
    ax.bar_label(b2,fontsize=8,padding=2)
    ax.set_xticks(x)
    ax.set_xticklabels(["fold 1","fold 2","fold 3"])
    ax.set_title(title,fontsize=9)
    ax.grid(axis="y",linestyle="--")
    ax.set_axisbelow(True)

axes[0].set_ylabel("high-cluster batches")
axes[0].set_ylim(0,112)
axes[0].annotate("fold 1 tested on 90 of 98\nwhile trained on 8",
                 xy=(0-w/2,90),xytext=(0.45,101),fontsize=8,color=C_TEST,
                 arrowprops=dict(arrowstyle="->",color=C_TEST,lw=0.8))
axes[1].annotate("code %d (%d batches) held out entirely;\n%d high-cluster batches remain"%(OOD,ood_n,tot_high),
                 xy=(1.0,60),xytext=(-0.35,88),fontsize=8,color="#444444")
axes[1].legend(loc="upper right",fontsize=8,framealpha=0.9)

fig.suptitle("Distribution of the 98 high-cluster batches across cross-validation folds",fontsize=10,y=1.02)
fig.tight_layout()
out=FIGURES/"fig5_fold_diagnostic.png"
fig.savefig(out)
plt.close(fig)
print("saved",out)

checks passed
 fold 1  broken test/train  90/  8   stratified test/train  22/ 12
 fold 2  broken test/train   6/ 92   stratified test/train   6/ 28
 fold 3  broken test/train   2/ 96   stratified test/train   6/ 28
saved C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis\results\figures\fig5_fold_diagnostic.png


## 14.6 Figure 2 rebuild - batches per product code

Rebuilds fig2 to remove the misleading "OOD threshold (50)" line from the earlier version.
That line drew a pure batch-count cutoff, which implied code 15 (64 batches) cleared a
threshold, when in fact code 15 was held out for being high-hardness, not for its count, and
two codes above 50 batches were never OOD candidates.

The real selection is encoded by colour instead of a line. Code 15 is the held-out OOD code.
The other four high-cluster codes are excluded from CA2 eligibility by hardness regime. The
five CA2-eligible codes (1, 13, 17, 21, 23) are the low-cluster codes with at least 50
batches, matching ca2_eligible_codes in extended_metrics.json. This overwrites
fig2_batches.png in place so the file stays reproducible from the notebook.

In [3]:
counts=lab["code"].value_counts()
mean_h=lab.groupby("code")["tbl_av_hardness"].mean()
ELIG=extended["ca2_eligible_codes"]

assert ELIG==[1,13,17,21,23]
assert set(HIGH)=={4,8,15,20,24}
for c in ELIG:
    assert counts[c]>=50 and c not in HIGH
assert int(counts[OOD])==64

order=counts.sort_values(ascending=True).index.tolist()
vals=[int(counts[c]) for c in order]

def cat(c):
    if c==OOD:return "ood"
    if c in HIGH:return "high"
    if c in ELIG:return "elig"
    return "other"

COL={"ood":"#c0392b","high":"#e08214","elig":"#2e6da4","other":"#b0b7c0"}
LBL={"ood":"code %d, held out for OOD (high hardness)"%OOD,
     "high":"other high-hardness codes (excluded)",
     "elig":"CA2-eligible (low hardness, >=50 batches)",
     "other":"low hardness, <50 batches"}
colors=[COL[cat(c)] for c in order]

print("category counts:")
for k in ["ood","high","elig","other"]:
    cs=[c for c in order if cat(c)==k]
    print("  %-6s %s"%(k,sorted(cs)))

fig,ax=plt.subplots(figsize=(7.2,7.6))
y=np.arange(len(order))
ax.barh(y,vals,color=colors,edgecolor="white",linewidth=0.5)
ax.set_yticks(y)
ax.set_yticklabels(order)
ax.set_ylabel("product code")
ax.set_xlabel("batches")
ax.set_title("Batches per product code, coloured by role in the study",fontsize=10)
ax.grid(axis="x",linestyle="--")
ax.set_axisbelow(True)

for yi,(c,v) in enumerate(zip(order,vals)):
    ax.text(v+2,yi,str(v),va="center",fontsize=7.5,color="#333333")

from matplotlib.patches import Patch
handles=[Patch(facecolor=COL[k],label=LBL[k]) for k in ["ood","high","elig","other"]]
ax.legend(handles=handles,loc="lower right",fontsize=8,framealpha=0.95)
ax.set_xlim(0,max(vals)*1.08)

fig.tight_layout()
out=FIGURES/"fig2_batches.png"
fig.savefig(out)
plt.close(fig)
print("saved (overwritten)",out)

category counts:
  ood    [15]
  high   [4, 8, 20, 24]
  elig   [1, 13, 17, 21, 23]
  other  [2, 3, 5, 6, 7, 9, 10, 11, 12, 14, 16, 18, 19, 22, 25]
saved (overwritten) C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis\results\figures\fig2_batches.png


## 14.7 Figure 6 - hardness error against product-code hardness

Each point is one held-out product code. Its out-of-fold predictions come from a model that
never trained on that code, so every point is a leave-one-code-out result. The x-axis is the
code's own mean hardness, read from the raw laboratory table at full precision. No reference
constant is used, so nothing here collides with the batch-count threshold in Figure 2.

The figure shows a valley. Error is lowest for codes whose hardness sits in the 45 to 55 N
band where most training data lies (mean out-of-fold RMSE 4.7 N), and rises on both sides:
toward the low-hardness codes at 36 to 39 N and sharply toward the high-hardness cluster near
90 N (mean RMSE 22.4 N outside the band). Error growing in both directions away from the
training centre is the signature of regression to the mean, and is stronger evidence than a
single upward trend would be.

For a one-number summary, Pearson correlation between hardness RMSE and absolute distance
from a 50 N reference is 0.94; it stays between 0.88 and 0.95 for any reference from 45 to
60 N, so the relationship does not depend on the exact centre. That number is reported in the
text only; the

In [4]:
loco=extended["leave_one_code_out"]
codes=sorted(loco.keys(),key=lambda x:int(x))
mh=np.array([lab[lab["code"]==int(c)]["tbl_av_hardness"].mean() for c in codes])
rmse_h=np.array([loco[c]["rmse"]["tbl_av_hardness"] for c in codes])
nb=np.array([loco[c]["n"] for c in codes])
is_hi=np.array([loco[c]["high_cluster"] for c in codes])

d50=np.abs(mh-50)
pear=stats.pearsonr(d50,rmse_h)
spear=stats.spearmanr(d50,rmse_h)
band=(mh>=45)&(mh<=55)

assert len(codes)==24
assert int(nb.sum())==941
assert OOD not in [int(c) for c in codes]
assert round(float(pear[0]),3)==0.941
assert round(float(spear[0]),3)==0.595
assert round(float(rmse_h[band].mean()),1)==4.7
print("codes %d  batches %d"%(len(codes),nb.sum()))
print("Pearson vs |meanH-50| r=%.4f p=%.2e"%(pear[0],pear[1]))
print("Spearman              r=%.4f p=%.4f"%(spear[0],spear[1]))
print("mean RMSE in 45-55 band %.2f  outside %.2f"%(rmse_h[band].mean(),rmse_h[~band].mean()))

def grp(i):
    if is_hi[i]:return "high"
    if mh[i]<45:return "low"
    return "centre"
COL={"high":"#c0392b","low":"#e08214","centre":"#2e6da4"}
LBL={"centre":"training band, 45-55 N","low":"low-hardness codes (11, 14, 16)","high":"high-hardness cluster (4, 8, 20, 24)"}

fig,ax=plt.subplots(figsize=(6.8,4.6))
ax.axvspan(45,55,color="#2e6da4",alpha=0.06,zorder=0)
for g in ["centre","low","high"]:
    m=np.array([grp(i)==g for i in range(len(codes))])
    ax.scatter(mh[m],rmse_h[m],s=18+nb[m]*1.1,c=COL[g],alpha=0.8,edgecolors="white",linewidths=0.6,label=LBL[g],zorder=3)

for i,c in enumerate(codes):
    if rmse_h[i]>10 or mh[i]<42:
        ax.annotate(c,(mh[i],rmse_h[i]),fontsize=7,xytext=(4,-2),textcoords="offset points",color="#333333")

ax.set_xlabel("product-code mean hardness (N)")
ax.set_ylabel("out-of-fold hardness RMSE (N)")
ax.set_title("Hardness error is lowest near the training band and rises on both sides",fontsize=10)
ax.grid(linestyle="--")
ax.set_axisbelow(True)
ax.legend(loc="upper center",fontsize=8,framealpha=0.95)
ax.text(0.02,0.97,"in-band mean RMSE 4.7 N\nout-of-band mean RMSE 22.4 N\n24 held-out codes, 941 batches",
        transform=ax.transAxes,ha="left",va="top",fontsize=7.5,
        bbox=dict(boxstyle="round,pad=0.4",facecolor="white",edgecolor="#cccccc"))

fig.tight_layout()
out=FIGURES/"fig6_hardness_vs_meanhardness.png"
fig.savefig(out)
plt.close(fig)
print("saved",out)

codes 24  batches 941
Pearson vs |meanH-50| r=0.9411 p=7.76e-12
Spearman              r=0.5948 p=0.0022
mean RMSE in 45-55 band 4.70  outside 22.35
saved C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis\results\figures\fig6_hardness_vs_meanhardness.png


## 14.8 Figure 7 - calibration

Empirical coverage against nominal coverage at three levels, read from extended_metrics.json.
The uncertainty scale was fitted per fold on the other two folds' out-of-fold predictions, so
no fold is calibrated on itself.

ECE values in the annotation are the 10-bin figures from mttrajnet_stratified.json, the
authoritative run. A 5-bin version exists as a CA2 deliverable and differs slightly; the bin
count is stated here so the two are not confused.

Three nominal levels are shown rather than one because they expose the weight RSD failure. At
the 90 percent level alone weight RSD reads 0.946 and looks acceptable. Across the three
levels its curve is nearly flat, 0.930 to 0.960, so the interval does not widen with the
nominal level as it should. Its ECE of 0.184 is an order of magnitude worse than the other
three targets, which sit near 0.02 to 0.04. Reporting only the 90 percent number would hide
this completely.

In [6]:
cov=extended["coverage"]
ece10={t:mtj["calibration"][t]["ece"] for t in ["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]}
TARGETS=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
NICE={"dissolution_av":"dissolution","tbl_av_hardness":"hardness",
      "tbl_rsd_weight":"weight RSD","fct_tensile":"tensile"}
COLT={"dissolution_av":"#2e6da4","tbl_av_hardness":"#2e8b57",
      "tbl_rsd_weight":"#c0392b","fct_tensile":"#8e6bab"}
nominal=np.array([0.80,0.90,0.95])

for t in TARGETS:
    assert t in cov
assert round(ece10["tbl_rsd_weight"],3)==0.184
assert round(cov["tbl_rsd_weight"]["picp80"],3)==0.930
assert round(cov["tbl_av_hardness"]["picp90"],3)==0.932
print("target        picp80 picp90 picp95  ece(10bin)")
for t in TARGETS:
    c=cov[t]
    print("%-13s %.3f  %.3f  %.3f  %.3f"%(NICE[t],c["picp80"],c["picp90"],c["picp95"],ece10[t]))

fig,ax=plt.subplots(figsize=(6.2,4.6))
ax.plot([0.78,0.97],[0.78,0.97],color="#999999",linestyle="--",linewidth=1.0,label="perfect calibration",zorder=2)
for t in TARGETS:
    c=cov[t]
    emp=np.array([c["picp80"],c["picp90"],c["picp95"]])
    ax.plot(nominal,emp,marker="o",markersize=5,linewidth=1.4,color=COLT[t],
            label="%s (ECE %.3f)"%(NICE[t],ece10[t]),zorder=3)

ax.annotate("weight RSD: nearly flat, 0.93 to 0.96;\ninterval does not scale with nominal level",
            xy=(0.90,0.946),xytext=(0.845,0.905),fontsize=7.5,color="#c0392b",
            arrowprops=dict(arrowstyle="->",color="#c0392b",lw=0.8))
ax.set_xticks(nominal)
ax.set_xticklabels(["0.80","0.90","0.95"])
ax.set_xlabel("nominal coverage")
ax.set_ylabel("empirical coverage (PICP)")
ax.set_title("Calibration across three nominal levels, 941 out-of-fold batches",fontsize=10)
ax.grid(linestyle="--")
ax.set_axisbelow(True)
ax.legend(loc="lower right",fontsize=7.5,framealpha=0.95)
fig.tight_layout()
out=FIGURES/"fig7_calibration.png"
fig.savefig(out)
plt.close(fig)
print("saved",out)

target        picp80 picp90 picp95  ece(10bin)
dissolution   0.770  0.857  0.897  0.021
hardness      0.846  0.932  0.957  0.038
weight RSD    0.930  0.946  0.960  0.184
tensile       0.790  0.841  0.865  0.033
saved C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis\results\figures\fig7_calibration.png


## 14.9 Figure 8 - multi-task against single-task

Single-task and multi-task RMSE on the four targets, on identical stratified folds with the
same cross-fitting setup, seed 42. Multi-task values are the authoritative notebook 08 run.
Significance is the Nadeau-Bengio corrected paired test; three folds gives low power, so
only dissolution reaches significance.

The left panel is RMSE per target, lower is better. Multi-task is at least as good on all
four targets and significantly better on dissolution (p=0.012). The right panel is the
parameter count: one multi-task model with 384k parameters against four single-task models
totalling 1.44m, a 3.73 times reduction. The multi-task model is not worse on any target and
uses just over a quarter of the parameters.

In [7]:
rq2=json.loads((RESULTS/"rq2_stratified.json").read_text())
TARGETS=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
NICE={"dissolution_av":"dissolution","tbl_av_hardness":"hardness",
      "tbl_rsd_weight":"weight RSD","fct_tensile":"tensile"}

single=[rq2["results"][t]["single_mean"] for t in TARGETS]
multi=[rq2["results"][t]["multi_mean"] for t in TARGETS]
pvals=[rq2["results"][t]["p"] for t in TARGETS]
p_multi=rq2["params"]["multi_task"]
p_single=rq2["params"]["four_single_task"]
reduction=rq2["params"]["reduction"]

assert rq2["results"]["dissolution_av"]["p"]==0.012
assert rq2["results"]["dissolution_av"]["significant"] is True
assert reduction==3.73
assert p_multi==384400 and p_single==1435408
assert [rq2["results"][t]["multi_mean"] for t in TARGETS]==[3.261,7.698,0.585,0.345]
print("target        single  multi   p      sig")
for t in TARGETS:
    d=rq2["results"][t]
    print("%-13s %.3f   %.3f  %.3f  %s"%(NICE[t],d["single_mean"],d["multi_mean"],d["p"],d["significant"]))
print("params multi %d single %d reduction %.2fx"%(p_multi,p_single,reduction))

C_SINGLE="#b0b7c0"
C_MULTI="#2e6da4"
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(9.4,4.2),gridspec_kw={"width_ratios":[2.3,1]})

x=np.arange(4)
w=0.38
b1=ax1.bar(x-w/2,single,w,color=C_SINGLE,label="single-task (4 models)")
b2=ax1.bar(x+w/2,multi,w,color=C_MULTI,label="multi-task (1 model)")
ax1.bar_label(b1,fmt="%.2f",fontsize=7.5,padding=2)
ax1.bar_label(b2,fmt="%.2f",fontsize=7.5,padding=2)
for i,p in enumerate(pvals):
    if p<0.05:
        top=max(single[i],multi[i])
        ax1.annotate("p=%.3f *"%p,xy=(i,top),xytext=(i,top*1.12),ha="center",fontsize=7.5,color="#c0392b",
                     arrowprops=dict(arrowstyle="-",color="#c0392b",lw=0.6))
ax1.set_xticks(x)
ax1.set_xticklabels([NICE[t] for t in TARGETS])
ax1.set_ylabel("out-of-fold RMSE (lower is better)")
ax1.set_title("(a) RMSE by target",fontsize=9)
ax1.grid(axis="y",linestyle="--")
ax1.set_axisbelow(True)
ax1.legend(loc="upper right",fontsize=8,framealpha=0.95)
ax1.set_ylim(0,max(single+multi)*1.2)

bars=ax2.bar([0,1],[p_single/1e6,p_multi/1e6],color=[C_SINGLE,C_MULTI],width=0.6)
ax2.bar_label(bars,fmt="%.2fM",fontsize=8,padding=2)
ax2.set_xticks([0,1])
ax2.set_xticklabels(["4 single\ntask","1 multi\ntask"])
ax2.set_ylabel("parameters (millions)")
ax2.set_title("(b) parameter count",fontsize=9)
ax2.grid(axis="y",linestyle="--")
ax2.set_axisbelow(True)
ax2.set_ylim(0,p_single/1e6*1.25)
ax2.annotate("%.2fx\nfewer"%reduction,xy=(1,p_multi/1e6),xytext=(1,p_single/1e6*0.6),ha="center",fontsize=9,color="#2e6da4",fontweight="bold")

fig.suptitle("Multi-task matches or beats four single-task models at a quarter of the parameters",fontsize=10,y=1.01)
fig.tight_layout()
out=FIGURES/"fig8_multitask_vs_singletask.png"
fig.savefig(out)
plt.close(fig)
print("saved",out)

target        single  multi   p      sig
dissolution   3.547   3.261  0.012  True
hardness      8.310   7.698  0.800  False
weight RSD    0.595   0.585  0.904  False
tensile       0.364   0.345  0.830  False
params multi 384400 single 1435408 reduction 3.73x
saved C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis\results\figures\fig8_multitask_vs_singletask.png


## 14.10 Figure 9 - uncertainty-stratified channel attribution

For each sensor channel, the ratio of its mean absolute SHAP attribution on the most
uncertain 20 percent of batches to the least uncertain 20 percent, split by per-target
epistemic uncertainty. A ratio above one means the channel matters more when the model is
uncertain. GradientExplainer, 32 background samples, fold 1 test set, reproduced identically
across two runs.

Only dissolution and hardness are shown, the two targets where the attribution shift is
significant and reproducible (Kendall tau 0.822, p 0.0004). Weight RSD and tensile are not
significant (tau 0.467, p 0.073) and are reported in the text, not here.

The hardness result is the interpretable finding: die fill is amplified 4.9 times and main
compression cylinder 2.8 times when the model is uncertain. These are the sensors that
physically govern tablet weight and compaction, so the model concentrates on the mechanism
that determines hardness exactly when it is least sure. Phase-level attribution is not shown:
the earlier phase pattern did not replicate after the fold correction and is discussed in the
text as a robustness observation.

In [9]:
rq3b=json.loads((RESULTS/"rq3b_all_targets_stratified.json").read_text())
CH_ORDER=["tbl_speed","fom","main_comp","tbl_fill","SREL","pre_comp","cyl_main","cyl_pre","stiffness","ejection"]
CH_NICE={"tbl_speed":"tbl speed","fom":"force of\ncompression","main_comp":"main\ncompression","tbl_fill":"die fill",
         "SREL":"SREL","pre_comp":"pre\ncompression","cyl_main":"cyl main","cyl_pre":"cyl pre",
         "stiffness":"stiffness","ejection":"ejection"}

diss=rq3b["results"]["dissolution_av"]
hard=rq3b["results"]["tbl_av_hardness"]
assert diss["kendall_tau"]==0.822 and diss["p"]==0.0004 and diss["significant"] is True
assert hard["kendall_tau"]==0.822 and hard["p"]==0.0004 and hard["significant"] is True
assert round(hard["channel_ratios"]["tbl_fill"],1)==4.9
assert round(hard["channel_ratios"]["cyl_main"],1)==2.8
assert rq3b["results"]["tbl_rsd_weight"]["significant"] is False
assert rq3b["results"]["fct_tensile"]["significant"] is False

d_ratios=[diss["channel_ratios"][c] for c in CH_ORDER]
h_ratios=[hard["channel_ratios"][c] for c in CH_ORDER]
print("channel        dissolution  hardness")
for c in CH_ORDER:
    print("%-14s %6.3f      %6.3f"%(c,diss["channel_ratios"][c],hard["channel_ratios"][c]))

order=sorted(range(len(CH_ORDER)),key=lambda i:-h_ratios[i])
labels=[CH_NICE[CH_ORDER[i]] for i in order]
dv=[d_ratios[i] for i in order]
hv=[h_ratios[i] for i in order]

C_DISS="#2e6da4"
C_HARD="#2e8b57"
y=np.arange(len(order))
w=0.38
fig,ax=plt.subplots(figsize=(7.2,5.4))
ax.axvline(1.0,color="#999999",linestyle="--",linewidth=1.0,zorder=1,label="equal attribution (ratio 1)")
b1=ax.barh(y+w/2,hv,w,color=C_HARD,label="hardness",zorder=3)
b2=ax.barh(y-w/2,dv,w,color=C_DISS,label="dissolution",zorder=3)
ax.bar_label(b1,fmt="%.1f",fontsize=7.5,padding=2)
ax.bar_label(b2,fmt="%.1f",fontsize=7.5,padding=2)

ax.set_yticks(y)
ax.set_yticklabels(labels,fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("attribution ratio, uncertain 20% over confident 20%")
ax.set_title("Which sensors the model leans on when uncertain (tau 0.822, p 0.0004)",fontsize=9.5)
ax.grid(axis="x",linestyle="--")
ax.set_axisbelow(True)
ax.legend(loc="lower right",fontsize=8,framealpha=0.95)
ax.set_xlim(0,max(hv)*1.15)
ax.text(0.60,0.62,"die fill 4.9x and main compression 2.8x\namplified when uncertain: the sensors that\ngovern tablet weight and compaction",
        transform=ax.transAxes,fontsize=7.5,color="#2e8b57",va="top",
        bbox=dict(boxstyle="round,pad=0.4",facecolor="white",edgecolor="#2e8b57",linewidth=0.8))

fig.tight_layout()
out=FIGURES/"fig9_channel_attribution.png"
fig.savefig(out)
plt.close(fig)
print("saved",out)

channel        dissolution  hardness
tbl_speed       1.078       1.384
fom             1.028       1.464
main_comp       0.681       1.582
tbl_fill        0.798       4.862
SREL            0.581       1.280
pre_comp        1.218       1.147
cyl_main        0.710       2.805
cyl_pre         0.984       1.517
stiffness       0.581       0.981
ejection        1.678       1.007
saved C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis\results\figures\fig9_channel_attribution.png
